# Phase 1: Data Acquisition & Corpus Construction

This notebook walks through **Phase 1** of the pipeline:
1. Scraping curated **TikTok Viral (`label=1`)** and **Non-Viral Commercial (`label=0`)** Spotify playlists (`config/playlists_viral.txt` and `config/playlists_non_viral.txt`).
2. Deduplicating tracks on normalized `(primary_artist, clean_title)` and removing any viral crossover songs from the non-viral set.
3. Fetching plain lyrics concurrently via the open **LRCLIB API** (`src/data/lrclib_fetcher.py`) with persistent local caching.
4. Creating the balanced dataset (up to 3,000 Viral / 3,000 Non-Viral) and stratified 80/20 train/test split.

In [2]:
import sys
from pathlib import Path

# Add repository root to sys.path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data.normalize import make_match_key, normalize_artist, normalize_title
from src.data.spotify_scraper import run_spotify_scraper
from src.data.lrclib_fetcher import fetch_lyrics_lrclib
from src.data.build_dataset import print_collection_report, build_balanced_splits

## 1. Sanity-Check Entity Normalization
Verify how `normalize_artist`, `normalize_title`, and `make_match_key` collapse remixes, sped-up versions, and `(feat. ...)` tags for strict exact matching.

In [3]:
examples = [
    ("Doja Cat", "Say So (feat. Nicki Minaj)"),
    ("FIFTY FIFTY", "Cupid - Twin Ver. (Sped Up)"),
    ("Lil Nas X & Jack Harlow", "INDUSTRY BABY (with Jack Harlow)"),
    ("The Weeknd", "Blinding Lights - 2020 Remix"),
]

pd.DataFrame([
    {
        "raw_artist": a,
        "raw_title": t,
        "norm_artist": normalize_artist(a),
        "norm_title": normalize_title(t),
        "match_key": make_match_key(a, t),
    }
    for a, t in examples
])

,raw_artist,raw_title,norm_artist,norm_title,match_key
0,Doja Cat,Say So (feat. Nicki Minaj),doja cat,say so,doja cat::say so
1,FIFTY FIFTY,Cupid - Twin Ver. (Sped Up),fifty fifty,cupid,fifty fifty::cupid
2,Lil Nas X & Jack Harlow,INDUSTRY BABY (with Jack Harlow),lil nas,industry baby,lil nas::industry baby
3,The Weeknd,Blinding Lights - 2020 Remix,the weeknd,blinding lights,the weeknd::blinding lights


## 2. Scrape Curated Spotify Playlists
Add playlist URLs/IDs to:
- `config/playlists_viral.txt`
- `config/playlists_non_viral.txt`

In [4]:
viral_sp, non_viral_sp = run_spotify_scraper(min_year=2020, max_year=2025, target_per_class=3000)


Scraping Viral Playlists (label=1, years=2020-2025, target=3000)
  [01/158] 'Best TikTok Viral Songs 2026 ⚡️' (0SRdjSDz4RsuozwHv1zhvr) -> 100 raw | 100 tracks | +100 new unique | Total: 100/3000
  [02/158] 'TIK TOK 2026 ⚡⚡ NOVEMBER HITS 2026' (4SWt6k4KUSNzmgRtCTzOKM) -> 100 raw | 100 tracks | +64 new unique | Total: 164/3000
  [03/158] 'Viral Ai Songs 2026 🔥 TikTok Ai Musi' (21AxwRqGwC05khfWy4cXLM) -> 25 raw | 25 tracks | +24 new unique | Total: 188/3000
  [04/158] 'TikTok Hits 2026 🔥 Trending & Viral ' (2wsA0CD8btzqPgUn1xH8U0) -> 99 raw | 99 tracks | +79 new unique | Total: 267/3000
  [05/158] 'Best Of TikTok 2020' (4yeF8WVBDCQV2VEQWZxaZJ) -> 100 raw | 100 tracks | +99 new unique | Total: 366/3000
  [06/158] 'VIRAL IN TIKTOK 2019-2023' (4ttiOjHqwjFRJu7JBVI5sN) -> 80 raw | 80 tracks | +75 new unique | Total: 441/3000
  [07/158] 'KIDS HITS 2026 💥 TikTok & Viral Tren' (3EaZ1fh9DShaeIK27DI685) -> 100 raw | 100 tracks | +56 new unique | Total: 497/3000
  [08/158] '2020 - 2021 TIKTOK ' (2V

## 3. Fetch Lyrics via LRCLIB & Build 80/20 Splits

In [5]:
combined_sp = pd.concat([viral_sp, non_viral_sp], ignore_index=True)
if not combined_sp.empty:
    matched_all, unmatched_all = fetch_lyrics_lrclib(combined_sp, max_workers=8)
    viral_matched = matched_all[matched_all["label"] == 1].copy()
    non_viral_matched = matched_all[matched_all["label"] == 0].copy()

    print_collection_report(combined_sp, matched_all, target_per_class=3000)
    full_df, train_df, test_df = build_balanced_splits(viral_matched, non_viral_matched, target_per_class=3000)
    display(full_df[["track_name", "primary_artist", "label_name", "lyrics"]].head(10))

[LRCLIB Cache] Loaded 158 previously matched songs from lrclib_lyrics_cache.parquet
[LRCLIB Fetcher] Class (label=1): have 125/3000 matched; scanning up to 7452 candidate tracks (workers=8)...


Matching LRCLIB (label=1): 100%|██████████| 2875/2875 [41:53<00:00,  1.14matched/s]  


[LRCLIB Fetcher] Class (label=1): found 2877 new lyrics (Total matched for class: 3002/3000).
[LRCLIB Fetcher] Class (label=0): have 30/3000 matched; scanning up to 4945 candidate tracks (workers=8)...


Matching LRCLIB (label=0): 100%|██████████| 2970/2970 [33:08<00:00,  1.49matched/s]  


[LRCLIB Fetcher] Class (label=0): found 3087 new lyrics (Total matched for class: 3117/3000).

PHASE 1 PROGRESS REPORT (LRCLIB Matched Lyrics Corpus)
  Viral (label=1)     :  7577 Spotify ->  3002 LRCLIB matched ( 39.6%) | Target: 3000 (0 more needed)
  Non-Viral (label=0) :  4975 Spotify ->  3117 LRCLIB matched ( 62.7%) | Target: 3000 (0 more needed)

[Dataset Saved] Full=6000 (Viral=3000, Non-Viral=3000) | Train (80%)=4800 | Test (20%)=1200 -> C:\University\Data Science\TxMM\txmm-project\data\processed


,track_name,primary_artist,label_name,lyrics
0,Lush Life,Zara Larsson,Viral,I live my day as if it was the last\nLive my d...
1,Espresso,Sabrina Carpenter,Viral,"Now he's thinkin' 'bout me every night, oh\nIs..."
2,Anxiety,Doechii,Viral,Anxiety\nKeep on trying me\nI feel it quietly\...
3,back to friends,sombr,Viral,Touch my body tender\n'Cause the feeling makes...
4,Dracula (with JENNIE),"Tame Impala, JENNIE",Viral,"Oh-oh-oh-oh-oh, oh-oh-oh-oh-oh (you and me)\nO..."
5,i like the way you kiss me,Artemas,Viral,I like the way you kiss me\nI like the way you...
6,Beauty And A Beat,"Justin Bieber, Nicki Minaj",Viral,"Yeah!\nYoung Money\nNicki Minaj\nJustin, grr!\..."
7,Sweater Weather,The Neighbourhood,Viral,And all I am is a man\nI want the world in my ...
8,Messy,Lola Young,Viral,You know I'm impatient\nSo why would you leave...
9,Heat Waves,Glass Animals,Viral,"Last night, all I think about is you\nDon't st..."
